# Power Demand Forecasting - Python Execution Walkthrough
This interactive notebook demonstrates the actual code used to clean your uploaded database, execute feature engineering, and train the Machine Learning Models step-by-step!

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

### Step 1: Loading the Dataset

In [2]:
filepath = '/Users/sagarsamrat/Downloads/powerdemand_5min_2021_to_2024_with weather.csv'
df = pd.read_csv(filepath)
display(df.head())

,Unnamed: 0,datetime,Power demand,temp,dwpt,rhum,wdir,wspd,pres,year,month,day,hour,minute,moving_avg_3
0,0,2021-01-01 00:30:00,2014.00,8.0,6.9,93.0,0.0,0.0,1017.0,2021,1,1,0,30,NaN
1,1,2021-01-01 00:35:00,2005.63,8.0,6.9,93.0,0.0,0.0,1017.0,2021,1,1,0,35,NaN
2,2,2021-01-01 00:40:00,1977.60,8.0,6.9,93.0,0.0,0.0,1017.0,2021,1,1,0,40,1999.076667
3,3,2021-01-01 00:45:00,1976.44,8.0,6.9,93.0,0.0,0.0,1017.0,2021,1,1,0,45,1986.556667
4,4,2021-01-01 00:50:00,1954.37,8.0,6.9,93.0,0.0,0.0,1017.0,2021,1,1,0,50,1969.470000


### Step 2: Data Cleaning

In [3]:
df['datetime'] = pd.to_datetime(df['datetime'])
df = df.sort_values('datetime')
df = df.drop_duplicates(subset=['datetime'])
df = df.set_index('datetime')
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])
df = df.dropna(subset=['Power demand'])
df = df.interpolate(method='time').bfill().ffill()
Q1 = df['Power demand'].quantile(0.25)
Q3 = df['Power demand'].quantile(0.75)
IQR = Q3 - Q1
df['Power demand'] = np.clip(df['Power demand'], Q1 - 1.5 * IQR, Q3 + 1.5 * IQR)
print("Cleaning Done! Validating missing values:\n", df.isnull().sum())

Cleaning Done! Validating missing values:
 Power demand    0
temp            0
dwpt            0
rhum            0
wdir            0
wspd            0
pres            0
year            0
month           0
day             0
hour            0
minute          0
moving_avg_3    0
dtype: int64


### Step 3: Feature Engineering

In [4]:
df['hour'] = df.index.hour
df['day'] = df.index.day
df['month'] = df.index.month
df['weekday'] = df.index.weekday
df['lag_24'] = df['Power demand'].shift(24)
df['lag_288'] = df['Power demand'].shift(288)
df['rolling_mean_12'] = df['Power demand'].shift(24).rolling(window=12).mean()
df = df.dropna()
display(df.tail())

,Power demand,temp,dwpt,rhum,wdir,wspd,pres,year,month,day,hour,minute,moving_avg_3,weekday,lag_1,lag_2,lag_24,rolling_mean_12
datetime,,,,,,,,,,,,,,,,,,
2024-12-12 00:10:00,2146.84,12.3,6.8,69.0,269.0,1.8,1019.4,2024,12,12,0,10,2174.893333,3,2154.75,2223.09,3446.53,3014.500000
2024-12-12 00:15:00,2116.66,12.3,6.8,69.0,269.0,1.8,1019.4,2024,12,12,0,15,2139.416667,3,2146.84,2154.75,3402.37,2922.173333
2024-12-12 00:20:00,2082.77,12.3,6.8,69.0,269.0,1.8,1019.4,2024,12,12,0,20,2115.423333,3,2116.66,2146.84,3348.17,2828.003333
2024-12-12 00:25:00,2059.17,12.3,6.8,69.0,269.0,1.8,1019.4,2024,12,12,0,25,2086.200000,3,2082.77,2116.66,3311.41,2734.331667
2024-12-12 00:30:00,2049.66,12.3,6.8,69.0,269.0,1.8,1019.4,2024,12,12,0,30,2063.866667,3,2059.17,2082.77,3297.00,2639.667500


### Step 4: Data Splitting (Time Series)

In [5]:
n = len(df)
train_df, val_df, test_df = df.iloc[:int(n*0.70)], df.iloc[int(n*0.70):int(n*0.85)], df.iloc[int(n*0.85):]
features = ['temp', 'rhum', 'wspd', 'hour', 'day', 'month', 'weekday', 'lag_24', 'lag_288', 'rolling_mean_12']
target = 'Power demand'
features = [f for f in features if f in df.columns]

X_train, y_train = train_df[features], train_df[target]
X_val, y_val = val_df[features], val_df[target]
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
print(f"Data Arrays Prepared: {len(X_train_scaled)} Training rows.")

Data Arrays Prepared: 275391 Training rows.


### Step 5: Training and Model Selection

In [6]:
def evaluate(name, y_true, y_pred):
    print(f"[{name}] -> RMSE: {np.sqrt(mean_squared_error(y_true, y_pred)):.2f} | R2: {r2_score(y_true, y_pred):.4f}")
    return np.sqrt(mean_squared_error(y_true, y_pred))

models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(n_estimators=50, max_depth=10, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42),
    "K-Neighbors": KNeighborsRegressor(n_neighbors=5, n_jobs=-1),
    "XGBoost": xgb.XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1)
}

best_m = None
best_rmse = float('inf')
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_val_scaled)
    rmse = evaluate(name, y_val, preds)
    if rmse < best_rmse:
        best_rmse = rmse
        best_m = name
print(f"\n🏆 Automatic Selection via Validation Metric indicates {best_m} performs best on this historical dataset!")

[Linear Regression] -> RMSE: 66.24 | R2: 0.9968


[Random Forest] -> RMSE: 68.17 | R2: 0.9966


[XGBoost] -> RMSE: 81.52 | R2: 0.9951

🏆 Automatic Selection via Validation Metric indicates Linear Regression performs best on this historical dataset!


In [ ]:
# Store predictions for analysis
model_predictions = {}
for name, model in models.items():
    preds = model.predict(X_test_scaled)
    model_predictions[name] = preds

import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (16, 12)

# ===== VISUALIZATION 1: Model Performance Comparison =====
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Model Performance Comparison on Test Set', fontsize=16, fontweight='bold')

# RMSE Comparison
rmse_values = []
model_names = list(model_predictions.keys())
for name in model_names:
    preds = model_predictions[name]
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    rmse_values.append(rmse)

axes[0, 0].barh(model_names, rmse_values, color='steelblue')
axes[0, 0].set_xlabel('RMSE (MW)', fontweight='bold')
axes[0, 0].set_title('Root Mean Squared Error (Lower is Better)')
axes[0, 0].invert_yaxis()
for i, v in enumerate(rmse_values):
    axes[0, 0].text(v, i, f' {v:.2f}', va='center')

# MAE Comparison
mae_values = []
for name in model_names:
    preds = model_predictions[name]
    mae = mean_absolute_error(y_test, preds)
    mae_values.append(mae)

axes[0, 1].barh(model_names, mae_values, color='coral')
axes[0, 1].set_xlabel('MAE (MW)', fontweight='bold')
axes[0, 1].set_title('Mean Absolute Error (Lower is Better)')
axes[0, 1].invert_yaxis()
for i, v in enumerate(mae_values):
    axes[0, 1].text(v, i, f' {v:.2f}', va='center')

# R² Score Comparison
r2_values = []
for name in model_names:
    preds = model_predictions[name]
    r2 = r2_score(y_test, preds)
    r2_values.append(r2)

axes[1, 0].barh(model_names, r2_values, color='mediumseagreen')
axes[1, 0].set_xlabel('R² Score', fontweight='bold')
axes[1, 0].set_title('R² Score (Higher is Better)')
axes[1, 0].invert_yaxis()
for i, v in enumerate(r2_values):
    axes[1, 0].text(v, i, f' {v:.4f}', va='center')

# Model Selection Summary
summary_text = f"""
🏆 BEST PERFORMING MODELS:

Best RMSE:  {model_names[np.argmin(rmse_values)]}
            RMSE = {min(rmse_values):.2f} MW

Best MAE:   {model_names[np.argmin(mae_values)]}
            MAE = {min(mae_values):.2f} MW

Best R²:    {model_names[np.argmax(r2_values)]}
            R² = {max(r2_values):.4f}

Selected:   {best_m}
            (Validation RMSE = {best_rmse:.2f} MW)
"""
axes[1, 1].text(0.05, 0.95, summary_text, transform=axes[1, 1].transAxes,
                fontsize=10, verticalalignment='top', family='monospace',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

print("\n✅ Model Performance Comparison Complete!")


In [ ]:
# Get predictions from best model for detailed analysis
best_model_obj = models[best_m]
test_predictions = best_model_obj.predict(X_test_scaled)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f'{best_m}: Predictions vs Actual on Test Set', fontsize=16, fontweight='bold')

# 1. Time Series Comparison
axes[0, 0].plot(y_test.values, label='Actual', linewidth=2, alpha=0.7)
axes[0, 0].plot(test_predictions, label='Predicted', linewidth=2, alpha=0.7)
axes[0, 0].set_xlabel('Test Sample Index', fontweight='bold')
axes[0, 0].set_ylabel('Power Demand (MW)', fontweight='bold')
axes[0, 0].set_title('Time Series: Actual vs Predicted')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# 2. Scatter Plot: Actual vs Predicted
axes[0, 1].scatter(y_test, test_predictions, alpha=0.5, s=30)
min_val = min(y_test.min(), test_predictions.min())
max_val = max(y_test.max(), test_predictions.max())
axes[0, 1].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
axes[0, 1].set_xlabel('Actual Demand (MW)', fontweight='bold')
axes[0, 1].set_ylabel('Predicted Demand (MW)', fontweight='bold')
axes[0, 1].set_title('Scatter: Actual vs Predicted')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# 3. Residuals (Errors)
residuals = y_test.values - test_predictions
axes[1, 0].plot(residuals, color='red', alpha=0.7, linewidth=1)
axes[1, 0].axhline(y=0, color='black', linestyle='--', linewidth=2)
axes[1, 0].fill_between(range(len(residuals)), residuals, 0, alpha=0.3, color='red')
axes[1, 0].set_xlabel('Test Sample Index', fontweight='bold')
axes[1, 0].set_ylabel('Error (MW)', fontweight='bold')
axes[1, 0].set_title('Prediction Residuals Over Time')
axes[1, 0].grid(alpha=0.3)

# 4. Error Distribution
axes[1, 1].hist(residuals, bins=50, edgecolor='black', color='skyblue', alpha=0.7)
axes[1, 1].axvline(x=0, color='red', linestyle='--', linewidth=2, label='Zero Error')
axes[1, 1].axvline(x=residuals.mean(), color='orange', linestyle='--', linewidth=2, label=f'Mean Error: {residuals.mean():.2f}')
axes[1, 1].set_xlabel('Error (MW)', fontweight='bold')
axes[1, 1].set_ylabel('Frequency', fontweight='bold')
axes[1, 1].set_title('Distribution of Prediction Errors')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Print error statistics
print(f"\n📊 PREDICTION ERROR STATISTICS ({best_m}):")
print(f"   Mean Error (Bias):       {residuals.mean():.4f} MW")
print(f"   Std Dev of Errors:       {residuals.std():.4f} MW")
print(f"   Min Error:               {residuals.min():.4f} MW")
print(f"   Max Error:               {residuals.max():.4f} MW")
print(f"   Mean Absolute Error:     {np.mean(np.abs(residuals)):.4f} MW")
print(f"   Root Mean Squared Error: {np.sqrt(np.mean(residuals**2)):.4f} MW")
print(f"   % Within ±5% of Actual:  {np.mean(np.abs(residuals/y_test.values) <= 0.05)*100:.2f}%")
print(f"   % Within ±10% of Actual: {np.mean(np.abs(residuals/y_test.values) <= 0.10)*100:.2f}%")


In [ ]:
# Feature Importance Analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Feature Importance Analysis', fontsize=16, fontweight='bold')

# For tree-based models, extract feature importance
if hasattr(best_model_obj, 'feature_importances_'):
    important_features = pd.DataFrame({
        'Feature': features,
        'Importance': best_model_obj.feature_importances_
    }).sort_values('Importance', ascending=True)
    
    axes[0].barh(important_features['Feature'], important_features['Importance'], color='seagreen')
    axes[0].set_xlabel('Importance Score', fontweight='bold')
    axes[0].set_title(f'{best_m}: Feature Importance')
    axes[0].grid(alpha=0.3, axis='x')
    
    # Add values on bars
    for i, v in enumerate(important_features['Importance']):
        axes[0].text(v, i, f' {v:.4f}', va='center')
else:
    axes[0].text(0.5, 0.5, f'Feature Importance\nnot available for\n{best_m}',
                 ha='center', va='center', fontsize=12, transform=axes[0].transAxes)
    axes[0].axis('off')

# Correlation with target
correlations = pd.DataFrame({
    'Feature': features,
    'Correlation': [train_df[f].corr(train_df['Power demand']) for f in features]
}).sort_values('Correlation', key=abs, ascending=True)

colors = ['red' if x < 0 else 'green' for x in correlations['Correlation']]
axes[1].barh(correlations['Feature'], correlations['Correlation'], color=colors, alpha=0.7)
axes[1].set_xlabel('Correlation with Demand', fontweight='bold')
axes[1].set_title('Feature Correlation with Target')
axes[1].axvline(x=0, color='black', linewidth=1)
axes[1].grid(alpha=0.3, axis='x')

# Add values on bars
for i, v in enumerate(correlations['Correlation']):
    axes[1].text(v, i, f' {v:.3f}', va='center')

plt.tight_layout()
plt.show()

print("\n🔍 TOP INFLUENCING FEATURES:")
if hasattr(best_model_obj, 'feature_importances_'):
    top_features = important_features.tail(5)
    print(f"\n{best_m} - Top 5 Most Important Features:")
    for idx, row in top_features.iterrows():
        print(f"   {row['Feature']:<20} : {row['Importance']:.4f}")

print(f"\nTop 5 Correlated Features with Power Demand:")
for idx, row in correlations.tail(5).iterrows():
    print(f"   {row['Feature']:<20} : {row['Correlation']:+.4f}")


In [ ]:
# Temporal Patterns Analysis
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Power Demand - Temporal Patterns', fontsize=16, fontweight='bold')

# 1. Hourly Pattern
hourly_avg = train_df.groupby('hour')['Power demand'].agg(['mean', 'std', 'min', 'max'])
axes[0, 0].plot(hourly_avg.index, hourly_avg['mean'], marker='o', linewidth=2, markersize=6, label='Mean', color='steelblue')
axes[0, 0].fill_between(hourly_avg.index, 
                        hourly_avg['mean'] - hourly_avg['std'],
                        hourly_avg['mean'] + hourly_avg['std'],
                        alpha=0.3, color='steelblue', label='±1 Std Dev')
axes[0, 0].set_xlabel('Hour of Day', fontweight='bold')
axes[0, 0].set_ylabel('Power Demand (MW)', fontweight='bold')
axes[0, 0].set_title('Average Demand Pattern by Hour')
axes[0, 0].set_xticks(range(0, 24, 2))
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# 2. Daily Pattern (by weekday)
day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
daily_avg = train_df.groupby('weekday')['Power demand'].agg(['mean', 'std'])
axes[0, 1].bar(range(7), daily_avg['mean'], yerr=daily_avg['std'], capsize=5, color='coral', alpha=0.7, edgecolor='black')
axes[0, 1].set_xlabel('Day of Week', fontweight='bold')
axes[0, 1].set_ylabel('Power Demand (MW)', fontweight='bold')
axes[0, 1].set_title('Average Demand by Day of Week')
axes[0, 1].set_xticks(range(7))
axes[0, 1].set_xticklabels(day_names)
axes[0, 1].grid(alpha=0.3, axis='y')

# 3. Monthly Pattern
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
monthly_avg = train_df.groupby('month')['Power demand'].agg(['mean', 'std', 'min', 'max'])
months = monthly_avg.index
axes[1, 0].plot(months, monthly_avg['mean'], marker='s', linewidth=2, markersize=7, label='Mean', color='mediumseagreen')
axes[1, 0].fill_between(months, monthly_avg['min'], monthly_avg['max'], alpha=0.2, color='mediumseagreen', label='Min-Max Range')
axes[1, 0].set_xlabel('Month', fontweight='bold')
axes[1, 0].set_ylabel('Power Demand (MW)', fontweight='bold')
axes[1, 0].set_title('Demand Trend by Month')
axes[1, 0].set_xticks(range(1, 13))
axes[1, 0].set_xticklabels(month_names, rotation=45)
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# 4. Distribution by Day Category
weekend_mask = train_df['weekday'].isin([5, 6])
weekday_demand = train_df[~weekend_mask]['Power demand']
weekend_demand = train_df[weekend_mask]['Power demand']

bp = axes[1, 1].boxplot([weekday_demand, weekend_demand], labels=['Weekdays', 'Weekends'],
                         patch_artist=True, notch=True)
for patch, color in zip(bp['boxes'], ['lightblue', 'lightcoral']):
    patch.set_facecolor(color)
axes[1, 1].set_ylabel('Power Demand (MW)', fontweight='bold')
axes[1, 1].set_title('Demand Distribution: Weekdays vs Weekends')
axes[1, 1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\n⏰ TEMPORAL INSIGHTS:")
print(f"\nHourly Pattern:")
print(f"   Peak Hour (Max Avg):     Hour {hourly_avg['mean'].idxmax():02d}:00 ({hourly_avg['mean'].max():.2f} MW)")
print(f"   Off-Peak Hour (Min Avg): Hour {hourly_avg['mean'].idxmin():02d}:00 ({hourly_avg['mean'].min():.2f} MW)")
print(f"\nWeekly Pattern:")
print(f"   Highest Day: {day_names[daily_avg['mean'].idxmax()]} ({daily_avg['mean'].max():.2f} MW)")
print(f"   Lowest Day:  {day_names[daily_avg['mean'].idxmin()]} ({daily_avg['mean'].min():.2f} MW)")
print(f"\nMonthly Pattern:")
print(f"   Peak Month:  {month_names[monthly_avg['mean'].idxmax()-1]} ({monthly_avg['mean'].max():.2f} MW)")
print(f"   Low Month:   {month_names[monthly_avg['mean'].idxmin()-1]} ({monthly_avg['mean'].min():.2f} MW)")
print(f"\nWeekday vs Weekend:")
print(f"   Weekday Avg:  {weekday_demand.mean():.2f} ± {weekday_demand.std():.2f} MW")
print(f"   Weekend Avg:  {weekend_demand.mean():.2f} ± {weekend_demand.std():.2f} MW")
print(f"   Difference:   {weekday_demand.mean() - weekend_demand.mean():.2f} MW")


In [ ]:
# Weather Impact Visualization
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Weather Factors Impact on Power Demand', fontsize=16, fontweight='bold')

# Temperature Impact
axes[0, 0].scatter(train_df['temp'], train_df['Power demand'], alpha=0.3, s=20, color='red')
axes[0, 0].set_xlabel('Temperature (°C)', fontweight='bold')
axes[0, 0].set_ylabel('Power Demand (MW)', fontweight='bold')
axes[0, 0].set_title(f'Temperature Impact (r={train_df["temp"].corr(train_df["Power demand"]):.3f})')
axes[0, 0].grid(alpha=0.3)
z = np.polyfit(train_df['temp'].dropna(), train_df.loc[train_df['temp'].notna(), 'Power demand'], 2)
p = np.poly1d(z)
axes[0, 0].plot(sorted(train_df['temp'].dropna()), p(sorted(train_df['temp'].dropna())), "r-", linewidth=2, alpha=0.8, label='Trend')
axes[0, 0].legend()

# Humidity Impact
axes[0, 1].scatter(train_df['rhum'], train_df['Power demand'], alpha=0.3, s=20, color='blue')
axes[0, 1].set_xlabel('Humidity (%)', fontweight='bold')
axes[0, 1].set_ylabel('Power Demand (MW)', fontweight='bold')
axes[0, 1].set_title(f'Humidity Impact (r={train_df["rhum"].corr(train_df["Power demand"]):.3f})')
axes[0, 1].grid(alpha=0.3)
z = np.polyfit(train_df['rhum'].dropna(), train_df.loc[train_df['rhum'].notna(), 'Power demand'], 2)
p = np.poly1d(z)
axes[0, 1].plot(sorted(train_df['rhum'].dropna()), p(sorted(train_df['rhum'].dropna())), "b-", linewidth=2, alpha=0.8, label='Trend')
axes[0, 1].legend()

# Wind Speed Impact
axes[0, 2].scatter(train_df['wspd'], train_df['Power demand'], alpha=0.3, s=20, color='green')
axes[0, 2].set_xlabel('Wind Speed', fontweight='bold')
axes[0, 2].set_ylabel('Power Demand (MW)', fontweight='bold')
axes[0, 2].set_title(f'Wind Speed Impact (r={train_df["wspd"].corr(train_df["Power demand"]):.3f})')
axes[0, 2].grid(alpha=0.3)
z = np.polyfit(train_df['wspd'].dropna(), train_df.loc[train_df['wspd'].notna(), 'Power demand'], 1)
p = np.poly1d(z)
axes[0, 2].plot(sorted(train_df['wspd'].dropna()), p(sorted(train_df['wspd'].dropna())), "g-", linewidth=2, alpha=0.8, label='Trend')
axes[0, 2].legend()

# Temperature-Humidity Interaction
temp_bins = pd.cut(train_df['temp'], bins=5)
humidity_bins = pd.cut(train_df['rhum'], bins=3)
interaction_data = train_df.groupby([temp_bins, humidity_bins])['Power demand'].mean().unstack()
im = axes[1, 0].imshow(interaction_data.values, cmap='YlOrRd', aspect='auto')
axes[1, 0].set_xlabel('Humidity Level', fontweight='bold')
axes[1, 0].set_ylabel('Temperature Level', fontweight='bold')
axes[1, 0].set_title('Temperature-Humidity Interaction')
axes[1, 0].set_xticklabels(['Low', 'Med', 'High'])
axes[1, 0].set_yticklabels(['Cold', 'Cool', 'Mild', 'Warm', 'Hot'])
plt.colorbar(im, ax=axes[1, 0], label='Avg Demand (MW)')

# Correlation Heatmap
weather_corr = train_df[['Power demand', 'temp', 'rhum', 'wspd']].corr()
im2 = axes[1, 1].imshow(weather_corr, cmap='coolwarm', vmin=-1, vmax=1)
axes[1, 1].set_xticks(range(len(weather_corr.columns)))
axes[1, 1].set_yticks(range(len(weather_corr.columns)))
axes[1, 1].set_xticklabels(['Demand', 'Temp', 'Humidity', 'Wind'], rotation=45)
axes[1, 1].set_yticklabels(['Demand', 'Temp', 'Humidity', 'Wind'])
axes[1, 1].set_title('Weather Variables Correlation')
for i in range(len(weather_corr)):
    for j in range(len(weather_corr)):
        text = axes[1, 1].text(j, i, f'{weather_corr.iloc[i, j]:.2f}',
                              ha="center", va="center", color="black", fontsize=9)
plt.colorbar(im2, ax=axes[1, 1], label='Correlation')

# Summary Statistics
summary_str = "WEATHER STATISTICS:\n"
summary_str += f"Temperature: {train_df['temp'].mean():.1f}±{train_df['temp'].std():.1f}°C\n"
summary_str += f"Humidity: {train_df['rhum'].mean():.1f}±{train_df['rhum'].std():.1f}%\n"
summary_str += f"Wind Speed: {train_df['wspd'].mean():.2f}±{train_df['wspd'].std():.2f}\n\n"
summary_str += "CORRELATIONS:\n"
summary_str += f"Temp ↔ Demand: {train_df['temp'].corr(train_df['Power demand']):.3f}\n"
summary_str += f"Humidity ↔ Dem: {train_df['rhum'].corr(train_df['Power demand']):.3f}\n"
summary_str += f"Wind ↔ Demand: {train_df['wspd'].corr(train_df['Power demand']):.3f}"

axes[1, 2].text(0.1, 0.9, summary_str, transform=axes[1, 2].transAxes,
               fontsize=10, verticalalignment='top', family='monospace',
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
axes[1, 2].axis('off')

plt.tight_layout()
plt.show()

print("\n🌡️ WEATHER IMPACT SUMMARY:")
print(f"\nCorrelation with Power Demand:")
print(f"   Temperature:  {train_df['temp'].corr(train_df['Power demand']):+.4f} (Strong {'positive' if train_df['temp'].corr(train_df['Power demand']) > 0 else 'negative'} correlation)")
print(f"   Humidity:     {train_df['rhum'].corr(train_df['Power demand']):+.4f}")
print(f"   Wind Speed:   {train_df['wspd'].corr(train_df['Power demand']):+.4f}")
print("\n✅ ALL ANALYSES COMPLETE!")


#### 6.4: Weather Impact Analysis

#### 6.3: Temporal Patterns in Demand

#### 6.2: Feature Importance Analysis

#### 6.1: Predictions vs Actual Analysis

### Step 6: Advanced Visualizations & Analysis